In [26]:
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까?: ')

#Step 3. 수집된 데이터를 저장할 파일 이름 입력 받기
f_dir = input("2.파일을 저장할 폴더명만 쓰세요(기본값:C:\\py_temp\\추출 후 파일 저장\\):")
if f_dir == '' :
    f_dir="C:\\py_temp\\추출 후 파일 저장\\"

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url ='https://korean.visitkorea.or.kr'
driver.get(url)
time.sleep(2)
driver.maximize_window()
time.sleep(2)

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'headerSearchBtn')
driver.find_element(By.ID,'headerSearchBtn').click( )
element.send_keys(query_txt)
element.send_keys("\n")
time.sleep(1)

#bs4로 본문내용 추출
from bs4 import BeautifulSoup
html_1 = driver.page_source #현재 페이지의 전체 소스코드를 다 가져오기
soup_1 = BeautifulSoup(html_1, 'html.parser')
time.sleep(2)

element = driver.find_element(By.ID,'inp_search_index')
driver.find_element(By.ID,'inp_search_index').click( )
element.send_keys(query_txt)
element.send_keys("\n")
time.sleep(2)

driver.find_element(By.LINK_TEXT,'동의').click()
time.sleep(2)

html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')
time.sleep(3)

driver.find_element(By.XPATH,'//*[@id="swiper-wrapper-1769c43bdbbd5bd1"]/div[3]/button').click()
time.sleep(2)



 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//*[@id="swiper-wrapper-1769c43bdbbd5bd1"]/div[3]/button"}
  (Session info: chrome=145.0.7632.119); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff72d33aa55
	0x7ff72d098630
	0x7ff72ce2d75d
	0x7ff72ce8862e
	0x7ff72ce8893c
	0x7ff72ced8d87
	0x7ff72ced591c
	0x7ff72ce79098
	0x7ff72ce79f83
	0x7ff72d367810
	0x7ff72d361afd
	0x7ff72d382c1a
	0x7ff72d0b3345
	0x7ff72d0bb81c
	0x7ff72d0a1924
	0x7ff72d0a1ad6
	0x7ff72d087e47
	0x7fff49b1e8d7
	0x7fff4a76c40c


In [27]:
#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력 받기
import math
total_cnt = soup_1.find('div','result_wrap').find('div','summary').find('strong').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
page_cnt = math.ceil(cnt / 8)
print('%s 건의 데이터를 수집하기 위해 %s 페이지의 게시물을 조회합니다.' %(cnt,page_cnt))
print("\n")

키워드 제주도여행 (으)로 총 9,095 건의 학위논문이 검색되었습니다
11 건의 데이터를 수집하기 위해 2 페이지의 게시물을 조회합니다.




In [30]:
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')
content_list = soup.find('div','spot_list').find_all('li')

In [31]:
len(content_list)

4

In [ ]:
#Step 9. 데이터 수집하기
no2=[] # 여행기사 번호 컬럼
title2=[ ] # 여행기사 제목 컬럼
name2=[] # 지자체이름
hash2=[ ] # 해시태그
cont2=[ ] # 본문내용

no = 1 # 여행기사 번호 초기값

for a in range(1,page_cnt+1) :
    print("\n")
    print("%s 페이지 내용 수집 시작합니다 =======================" %a)
    time.sleep(3)
    
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    content_list = soup.find('div','spot_list').find_all('li')

    for i in content_list:
        # 기사 제목 체크하기
        try:
            title=i.find('div','tit_wrap').get_text().strip()
        except :
            continue
        else :
            # 1.여행기사 번호
            print("\n")
            print("%s 번째 정보를 추출하고 있습니다============" %no)
            no2.append(no)
            print("1.번호 : %s" %no)

            # 2. 여행기사 제목
            title2.append(title.strip())
            print("2.제목 : %s" %title.strip())
            
            # 3. 지자체이름
            try :
                name=i.find("div","area_wrap").find('span').get_text().replace("\n","").strip()
            except :
                name='지자체이름이 없습니다'
                name2.append(name)
                print("3.지자체이름 : %s" %name)
            else :
                name2.append(name)
                print("3.지자체이름 : %s" %name)

            # 4. 해시태그
            try :
                hash=i.find('span]','keyword').find_all('span').get_text().replace("#"," #")
            except :
                hash = '해시태그가 없습니다'
                print("4.해시태그 : %s" %hash.strip())
                hash2.append(hash.strip())
            else :
                hash2.append(hash.strip())
                print("4.해시태그 : %s" %hash.strip())

            # 상세 정보 페이지로 들어감
            driver.find_element(By.XPATH,'//*[@id="contents"]/div/div[2]/div[2]/div[3]/div/ul/li[1]').click()
            time.sleep(3)

            html_1 = driver.page_source
            soup_1 = BeautifulSoup(html_1, 'html.parser')
            time.sleep(3)


            # 5. 본문내용    
            try :
                cont=soup_1.find('div','inr').get_text().strip()
            except :
                cont = '본문내용이 없습니다'
                print("5.본문내용 : %s" %cont.strip())
                cont2.append(cont.strip())
            else :
                cont2.append(cont.strip())
                print("5.본문내용 : %s" %cont.strip())

            driver.back() # 이전 페이지로 돌아가기

            time.sleep(3)

            no += 1

            if no > cnt :
                break

    a += 1
    b = str(a)

    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
    except :
        driver.find_element(By.LINK_TEXT,'다음').click()

print("요청하신 작업이 모두 완료되었습니다")



1 페이지 내용 수집 시작합니다 =======================


AttributeError: 'NoneType' object has no attribute 'find_all'

In [ ]:

# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
# 현재 날짜와 시간으로 폴더 만들고 파일 이름 설정하기
import os

n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+'응답소'+'-'+s+'-'+'민원')

fc_name = f_dir+'응답소'+'-'+s+'-'+'민원'+'\\'+'응답소'+'-'+s+'-'+'민원'+'.csv'
fx_name = f_dir+'응답소'+'-'+s+'-'+'민원'+'\\'+'응답소'+'-'+s+'-'+'민원'+'.xlsx'

# 데이터 프레임 생성 후 xlsx, csv 형식으로 저장하기
import pandas as pd

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['공개일']=pd.Series(opend2)
df['상담내역']=pd.Series(story2)

# xls 형태로 저장하기
df.to_excel(fx_name,index=False)

# csv 형태로 저장하기
df.to_csv(fc_name,index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

In [14]:
for i in content_list:
        # 1. 기사 제목 체크하기 (유령 데이터 거르기)
        try:
            title = i.find('div','tit_wrap').find('strong').get_text().strip()
        except:
            continue
            
        # 1.여행기사 번호
        print("\n")
        print("%s 번째 정보를 추출하고 있습니다============" %no)
        no2.append(no)
        print("1.번호 : %s" %no)

        # 2. 여행기사 제목
        title2.append(title)
        print("2.제목 : %s" %title)
        
        # 3. 지자체이름
        try:
            name = i.find("div","tit_wrap").find('span','area').get_text().strip()
        except:
            name = '지자체이름이 없습니다'
            name2.append(name)
            print("3.지자체이름 : %s" %name)

        time.sleep(1)

        # 💡 [핵심 1] 상세정보 들어가기: 허공 클릭 대신, 진짜 링크를 찾아 실행!
        try:
            # 보통 a 태그 안에 이동하는 자바스크립트나 링크가 들어있습니다.
            url_1 = i.find('a')['href'] 
            driver.execute_script(url_1) 
            time.sleep(3) # 페이지 뜰 때까지 대기
        except Exception as e:
            print(f"상세페이지 진입 실패: {e}")
            continue # 진입 실패 시 다음 기사로 넘어감

        # ---------------- 여기서부터 상세 페이지 ----------------
        html_1 = driver.page_source
        soup_1 = BeautifulSoup(html_1, 'html.parser')
        
        # 💡 [핵심 2] 해시태그: find_all() 대신 ul 박스 전체의 텍스트를 한 번에 추출!
        try:
            # 클래스 이름이 'tag clfix'에서 바뀔 수도 있으니 'tag'만으로 찾는 것이 안전합니다.
            hash_box = soup_1.find('ul', class_='tag')
            if hash_box:
                hash = hash_box.get_text(strip=True) # li를 다 합쳐서 텍스트만 뽑아줌
            else:
                hash = '해시태그가 없습니다'
        except:
            hash = '해시태그가 없습니다'
            
        hash2.append(hash)
        print("4.해시태그 : %s" %hash)

        # 5. 본문내용    
        try:
            cont = soup_1.find('div','txt_p').get_text().strip()
        except:
            cont = '본문내용이 없습니다'
            
        cont2.append(cont)
        print("5.본문내용 : %s..." %cont[:30]) # 화면 깔끔하게 30자만 출력

        driver.back() # 이전 페이지로 돌아가기
        time.sleep(3) # 목록 뜰 때까지 대기

        no += 1

        if no > cnt:
            break



12 번째 정보를 추출하고 있습니다============
1.번호 : 12
2.제목 : [제주도민추천 '갓성비' 제주여행 1탄] 제주여행물가 팩트 체크
3.지자체이름 : 지자체이름이 없습니다
4.해시태그 : #김녕청굴물#돈내코#수산유원지#원앙폭포#제주가볼만한곳#제주돈내코#제주돌문화공원#제주여행#제주원앙폭포#청굴물
5.본문내용 : 제주물가가 해외보다 비싸서 제주 갈 바에야 해외여행 가...
